In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f


## 1) SparkSession

Crearemos la `SparkSession` para ejecutar DataFrames y SQL.


In [2]:
spark = (SparkSession.builder
         .appName("ClaseSpark_SQL_DataFrames_TusDatyos")
         .master("local[*]")  # si necesitas forzar modo local
         .config("spark.sql.shuffle.partitions", "64")
         .getOrCreate())

spark

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/29 22:28:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



## 2) Carga de tus datasets (robusta)


In [3]:
trump = spark.read.json('../working_dir/trump_tweets/donald_data.json')
trump.printSchema()

root
 |-- contributors: string (nullable = true)
 |-- coordinates: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- display_text_range: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- entities: struct (nullable = true)
 |    |-- hashtags: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- indices: array (nullable = true)
 |    |    |    |    |-- element: long (containsNull = true)
 |    |    |    |-- text: string (nullable = true)
 |    |-- media: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- display_url: string (nullable = true)
 |    |    |    |-- expanded_url: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- id_str: string (nullable = true)
 |    |    |    |-- indices: array (nullable = true)
 |    |    |    |    |-- element: long (containsNull = true)
 |    |    |    |-- media_url: string (nulla

25/09/29 22:29:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
trump.select("extended_entities").show(1,False, True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


## Seleccionando del dataframe

In [5]:
trump_clean = trump.select(
  'user.screen_name',
  'created_at',
  'full_text',
  'retweet_count',
  'favorite_count')
trump_clean.show()

+---------------+--------------------+--------------------+-------------+--------------+
|    screen_name|          created_at|           full_text|retweet_count|favorite_count|
+---------------+--------------------+--------------------+-------------+--------------+
|realDonaldTrump|Tue Sep 11 20:16:...|The safety of Ame...|         7270|         27341|
|realDonaldTrump|Tue Sep 11 16:48:...|Small Business Op...|         8520|         31085|
|realDonaldTrump|Tue Sep 11 15:32:...|#NeverForget #Sep...|         9641|         33968|
|realDonaldTrump|Tue Sep 11 12:58:...|17 years since Se...|        17952|         73372|
|realDonaldTrump|Tue Sep 11 12:24:...|Departing Washing...|        12541|         52227|
|realDonaldTrump|Tue Sep 11 11:59:...|Rudy Giuliani did...|        19674|         90609|
|realDonaldTrump|Tue Sep 11 11:41:...|“ERIC Holder coul...|        10760|         43470|
|realDonaldTrump|Tue Sep 11 11:19:...|New Strzok-Page t...|        18090|         64190|
|realDonaldTrump|Tue 

In [6]:
trump_clean = trump.select(
  'user.screen_name',
  'created_at',
  f.encode('full_text', 'ascii').cast('string').alias('text'),
  'retweet_count',
  'favorite_count')
trump_clean.show()

+---------------+--------------------+--------------------+-------------+--------------+
|    screen_name|          created_at|                text|retweet_count|favorite_count|
+---------------+--------------------+--------------------+-------------+--------------+
|realDonaldTrump|Tue Sep 11 20:16:...|The safety of Ame...|         7270|         27341|
|realDonaldTrump|Tue Sep 11 16:48:...|Small Business Op...|         8520|         31085|
|realDonaldTrump|Tue Sep 11 15:32:...|#NeverForget #Sep...|         9641|         33968|
|realDonaldTrump|Tue Sep 11 12:58:...|17 years since Se...|        17952|         73372|
|realDonaldTrump|Tue Sep 11 12:24:...|Departing Washing...|        12541|         52227|
|realDonaldTrump|Tue Sep 11 11:59:...|Rudy Giuliani did...|        19674|         90609|
|realDonaldTrump|Tue Sep 11 11:41:...|?ERIC Holder coul...|        10760|         43470|
|realDonaldTrump|Tue Sep 11 11:19:...|New Strzok-Page t...|        18090|         64190|
|realDonaldTrump|Tue 

## Show

In [7]:
trump_clean.show(n=3, truncate=False, vertical=True) 

-RECORD 0---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 screen_name    | realDonaldTrump                                                                                                                                                                                                   
 created_at     | Tue Sep 11 20:16:49 +0000 2018                                                                                                                                                                                    
 text           | The safety of American people is my absolute highest priority. Heed the directions of your State and Local Officials. Please be prepared, be careful and be SAFE! https://t.co/YP7ssITwW9 https://t.co/LZIUCgdPTH 
 retweet_count  | 7270                                                              

## Esquema

In [8]:
trump_clean.printSchema()

root
 |-- screen_name: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- text: string (nullable = true)
 |-- retweet_count: long (nullable = true)
 |-- favorite_count: long (nullable = true)



# Exploración
#### Los 3 tweets más retuiteados

In [10]:
top3_rt = trump_clean.orderBy(f.desc("retweet_count")).limit(3)
top3_rt.show(n=3, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 screen_name    | realDonaldTrump                                                                                                                                                                                                                                                                              
 created_at     | Sun Nov 12 00:48:01 +0000 2017                                                                                                                                                                                                                                                               
 text           | Why would Kim Jong-un insult me by calling me "old," when I would NEVE

## Con Spark SQL

In [13]:
trump_clean.createOrReplaceTempView("tweets")
# Ejecuta la consulta SQL para obtener los 3 tweets más retuiteados
query = """
    SELECT *
    FROM tweets
    ORDER BY retweet_count DESC
    LIMIT 3
"""

top_3_tweets = spark.sql(query)

# Muestra los 3 tweets más retuiteados
top_3_tweets.show(n=3, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 screen_name    | realDonaldTrump                                                                                                                                                                                                                                                                              
 created_at     | Sun Nov 12 00:48:01 +0000 2017                                                                                                                                                                                                                                                               
 text           | Why would Kim Jong-un insult me by calling me "old," when I would NEVE

## Agregaciones en SparkSQL

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)
 |-- _c22: string (nullable = true)
 |-- _c23: string (nullable = true)
 |-- _c24: string (nullable = true)
 |-- _c25: string (nullable = true)
 |-- _c26: string (nullable = true)
 |-- _c27: string (nullable = tru

In [12]:
nba = spark.read.csv('../working_dir/nba/games_details.csv',   header = True, 
  inferSchema = True ) 
nba.printSchema()

[Stage 9:==>                                                      (1 + 19) / 20]

root
 |-- GAME_ID: integer (nullable = true)
 |-- TEAM_ID: integer (nullable = true)
 |-- TEAM_ABBREVIATION: string (nullable = true)
 |-- TEAM_CITY: string (nullable = true)
 |-- PLAYER_ID: integer (nullable = true)
 |-- PLAYER_NAME: string (nullable = true)
 |-- START_POSITION: string (nullable = true)
 |-- COMMENT: string (nullable = true)
 |-- MIN: string (nullable = true)
 |-- FGM: double (nullable = true)
 |-- FGA: double (nullable = true)
 |-- FG_PCT: double (nullable = true)
 |-- FG3M: double (nullable = true)
 |-- FG3A: double (nullable = true)
 |-- FG3_PCT: double (nullable = true)
 |-- FTM: double (nullable = true)
 |-- FTA: double (nullable = true)
 |-- FT_PCT: double (nullable = true)
 |-- OREB: double (nullable = true)
 |-- DREB: double (nullable = true)
 |-- REB: double (nullable = true)
 |-- AST: double (nullable = true)
 |-- STL: double (nullable = true)
 |-- BLK: double (nullable = true)
 |-- TO: double (nullable = true)
 |-- PF: double (nullable = true)
 |-- PTS: dou

DataFrame[GAME_ID: int, TEAM_ID: int, TEAM_ABBREVIATION: string, TEAM_CITY: string, PLAYER_ID: int, PLAYER_NAME: string, START_POSITION: string, COMMENT: string, MIN: string, FGM: double, FGA: double, FG_PCT: double, FG3M: double, FG3A: double, FG3_PCT: double, FTM: double, FTA: double, FT_PCT: double, OREB: double, DREB: double, REB: double, AST: double, STL: double, BLK: double, TO: double, PF: double, PTS: double, PLUS_MINUS: double]

## Preguntas:
#### 1. ¿Cuál es la máxima cantidad de puntos anotados por un jugador en un solo juego?
#### 2.	En promedio, ¿cuántos puntos anota un jugador por juego?
#### 3.	¿Qué precisión tienen en promedio por juego todos los jugadores de la NBA en lanzamientos en tiempo de juego? (FG_PCT)
#### 4.	Si filtramos sólo los juegos de Kobe Bryant, ¿cuántos puntos hace por juego en promedio?, y ¿qué precisión tiene él en promedio por juego en lanzamientos? (FG_PCT)
#### 5.	Volvamos a comparar a Kobe con el promedio de jugadores, pero ahora con la cantidad promedio de intentos (FGA) de canasta.
#### 6.	Volvamos a calcular las tres estadísticas, pero ahora filtrando para todos los jugadores que juegen la misma posición que Kobe.
#### 7.	Calculemos la columna “minutes” del DataFrame que contenga la cantidad de minutos jugados sin considerar los segundos. Esto lo podemos lograr tomando sólo los primeros dos dígitos del string que representan el minuto con la función substring().
#### 8. Comparemos el promedio de esta estadística para los tres subconjuntos del dataset con los que estamos trabajando: total de jugadores, guardas (START_POSITION = “G”), y Kobe Bryant.
#### 9. ¿Será Kobe el jugador que tenga más alta esta estadística en un solo juego?

## Respuestas
#### 1. ¿Cuál es la máxima cantidad de puntos anotados por un jugador en un solo juego?


In [28]:
nba.groupBy("GAME_ID","PLAYER_NAME")\
    .agg(f.max("pts").alias("Max_Ptos"))\
    .orderBy(f.desc("Max_Ptos"))\
    .show()

[Stage 16:=====>                                                  (2 + 18) / 20]

+--------+---------------+--------+
| GAME_ID|    PLAYER_NAME|Max_Ptos|
+--------+---------------+--------+
|20500591|    Kobe Bryant|    81.0|
|21601076|   Devin Booker|    70.0|
|20600977|    Kobe Bryant|    65.0|
|20300927|  Tracy McGrady|    62.0|
|22000092|  Stephen Curry|    62.0|
|20500359|    Kobe Bryant|    62.0|
|21300640|Carmelo Anthony|    62.0|
|21900652| Damian Lillard|    61.0|
|21901300| Damian Lillard|    61.0|
|21800710|   James Harden|    61.0|
|20800709|    Kobe Bryant|    61.0|
|21801084|   James Harden|    61.0|
|21300893|   LeBron James|    61.0|
|20400742|  Allen Iverson|    60.0|
|20601016|    Kobe Bryant|    60.0|
|21900125| Damian Lillard|    60.0|
|20600355| Gilbert Arenas|    60.0|
|21501228|    Kobe Bryant|    60.0|
|21800225|   Kemba Walker|    60.0|
|22000950|   Jayson Tatum|    60.0|
+--------+---------------+--------+
only showing top 20 rows



#### 2. En promedio, ¿cuántos puntos anota un jugador por juego?

In [31]:
nba.groupBy("PLAYER_NAME")\
    .agg(f.avg("pts").alias("Avg_Ptos"))\
    .orderBy(f.desc("Avg_Ptos"))\
    .show()

[Stage 25:==>                                                     (1 + 19) / 20]

+------------------+------------------+
|       PLAYER_NAME|          Avg_Ptos|
+------------------+------------------+
|      Kevin Durant|26.827046918123276|
|      LeBron James| 26.75803517283202|
|       Kobe Bryant|26.621463414634146|
|     Allen Iverson|25.863340563991322|
|   Zion Williamson|25.619565217391305|
|       Luka Doncic| 25.51818181818182|
|      James Harden| 24.60861423220974|
|    Damian Lillard|24.293519695044473|
|       Joel Embiid|24.135313531353134|
|     Stephen Curry| 24.05095541401274|
|        Trae Young|23.690582959641254|
|     Anthony Davis| 23.66923076923077|
|  Donovan Mitchell| 23.28930817610063|
| Russell Westbrook| 22.85131459655485|
|   Carmelo Anthony|22.683937823834196|
|      Kyrie Irving|22.571005917159763|
|      Devin Booker|22.548974943052393|
|Karl-Anthony Towns|22.401360544217688|
|       Dwyane Wade|21.633564280215552|
|      Bradley Beal|21.633477633477632|
+------------------+------------------+
only showing top 20 rows



#### 3.	¿Qué precisión tienen en promedio por juego todos los jugadores de la NBA en lanzamientos en tiempo de juego? (FG_PCT)

In [35]:
nba.agg(f.avg("FG_PCT").alias("FG_PCT_AVG"))\
    .show()

+------------------+
|        FG_PCT_AVG|
+------------------+
|0.4159210476805598|
+------------------+



#### 4.	Si filtramos sólo los juegos de Kobe Bryant, ¿cuántos puntos hace por juego en promedio?, y ¿qué precisión tiene él en promedio por juego en lanzamientos? (FG_PCT)

In [40]:
kobe = nba.filter(nba["PLAYER_NAME"] == "Kobe Bryant")
kobe.select(f.avg(kobe["PTS"]).alias("AVG_PTS"),
            f.avg(kobe["FG_PCT"]).alias("FG_PCT_AVG")).show()

+------------------+-------------------+
|           AVG_PTS|         FG_PCT_AVG|
+------------------+-------------------+
|26.621463414634146|0.44143512195121937|
+------------------+-------------------+



#### 5. Volvamos a comparar a Kobe con el promedio de jugadores, pero ahora con la cantidad promedio de intentos (FGA) de canasta.

In [47]:
avg_FGA = nba.agg(f.avg(nba["FGA"]).alias("FGA"))
avg_FGA.show()
kobe_avg_FGA = kobe.agg(f.avg(kobe["FGA"]).alias("FGA_Kobe"))
kobe_avg_FGA.show()

+------------------+
|               FGA|
+------------------+
|7.8798666348576445|
+------------------+

+------------------+
|          FGA_Kobe|
+------------------+
|20.692682926829267|
+------------------+



#### 6.	Volvamos a calcular las tres estadísticas, pero ahora filtrando para todos los jugadores que juegen la misma posición que Kobe.

In [60]:
kobe_position = kobe.select("START_POSITION").first()[0]
kobe_position
jugadores_misma_pos = nba.filter(nba["START_POSITION"] == kobe_position)
#Quitar a Kobe
jugadores_misma_pos = jugadores_misma_pos.filter("player_name != 'Kobe Bryant'")

kobe.agg(f.avg(kobe["PTS"]).alias("Kobe_PTS"),
         f.avg(kobe["FG_PCT"]).alias("Kobe_FG_PCT"),
         f.avg(kobe["FGA"]).alias("Kobe_FGA")).show()

jugadores_misma_pos.agg(f.avg(jugadores_misma_pos["PTS"]).alias("Misma_pos_PTS"),
         f.avg(jugadores_misma_pos["FG_PCT"]).alias("Misma_pos_FG_PCT"),
         f.avg(jugadores_misma_pos["FGA"]).alias("Misma_FGA")).show()


+------------------+-------------------+------------------+
|          Kobe_PTS|        Kobe_FG_PCT|          Kobe_FGA|
+------------------+-------------------+------------------+
|26.621463414634146|0.44143512195121937|20.692682926829267|
+------------------+-------------------+------------------+

+------------------+-------------------+------------------+
|     Misma_pos_PTS|   Misma_pos_FG_PCT|         Misma_FGA|
+------------------+-------------------+------------------+
|14.821590264473247|0.42875309051599936|12.179405122496815|
+------------------+-------------------+------------------+



#### 7. Calculemos la columna “minutes” del DataFrame que contenga la cantidad de minutos jugados sin considerar los segundos. Esto lo podemos lograr tomando sólo los primeros dos dígitos del string que representan el minuto con la función substring().

In [66]:
nba_minutes = nba.withColumn("minutes",f.substring(f.col("MIN"),1,2).cast('int'))
nba_minutes.select("minutes").show()

+-------+
|minutes|
+-------+
|     34|
|     25|
|     12|
|     19|
|     29|
|     23|
|     23|
|     20|
|     16|
|     10|
|   NULL|
|   NULL|
|   NULL|
|   NULL|
|     21|
|     23|
|     26|
|     23|
|     28|
|     19|
+-------+
only showing top 20 rows



#### 8. Comparemos el promedio de esta estadística para los tres subconjuntos del dataset con los que estamos trabajando: total de jugadores, guardas (START_POSITION = “G”), y Kobe Bryant.


In [74]:
kobe_minutes = kobe.withColumn("minutes",f.substring(f.col("MIN"),1,2).cast('int'))
jugadores_misma_pos_minutes = jugadores_misma_pos.withColumn("minutes",f.substring(f.col("MIN"),1,2).cast('int'))
nba_minutes.agg(f.avg(f.col('minutes')).alias("minutes_todos")).show()
jugadores_misma_pos_minutes.agg(f.avg(f.col('minutes')).alias("minutes_misma_pos")).show()
kobe_minutes.agg(f.avg(f.col('minutes')).alias("minutes_kobe")).show()


+------------------+
|     minutes_todos|
+------------------+
|25.527026106851515|
+------------------+

+------------------+
| minutes_misma_pos|
+------------------+
|31.885587516288485|
+------------------+

+-----------------+
|     minutes_kobe|
+-----------------+
|36.76320939334638|
+-----------------+



#### 9. ¿Será Kobe el jugador que tenga más alta esta estadística en un solo juego?

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `FGA_per_minute` cannot be resolved. Did you mean one of the following? [`minutes`, `GAME_ID`, `FG3_PCT`, `FGA`, `FG_PCT`].;
'Aggregate [player_name#314, start_position#315], [player_name#314, start_position#315, avg('FGA_per_minute) AS Promedio_FGA_Todos#2953]
+- Project [GAME_ID#309, TEAM_ID#310, TEAM_ABBREVIATION#311, TEAM_CITY#312, PLAYER_ID#313, PLAYER_NAME#314, START_POSITION#315, COMMENT#316, MIN#317, FGM#318, FGA#319, FG_PCT#320, FG3M#321, FG3A#322, FG3_PCT#323, FTM#324, FTA#325, FT_PCT#326, OREB#327, DREB#328, REB#329, AST#330, STL#331, BLK#332, ... 5 more fields]
   +- Relation [GAME_ID#309,TEAM_ID#310,TEAM_ABBREVIATION#311,TEAM_CITY#312,PLAYER_ID#313,PLAYER_NAME#314,START_POSITION#315,COMMENT#316,MIN#317,FGM#318,FGA#319,FG_PCT#320,FG3M#321,FG3A#322,FG3_PCT#323,FTM#324,FTA#325,FT_PCT#326,OREB#327,DREB#328,REB#329,AST#330,STL#331,BLK#332,... 4 more fields] csv
